In [4]:
import os
from dotenv import load_dotenv

load_dotenv()

token = os.environ.get("OPENAI_API_KEY")

if token:
    masked = f"{token[:4]}...{token[-4:]}" if len(token) > 8 else "****"
    print(f"OPENAI_API_KEY loaded ({len(token)} characters): {masked}")
else:
    print("OPENAI_API_KEY not found. Check that .env exists in this directory, "
          "the key is spelled exactly 'OPENAI_API_KEY', and load_dotenv() ran without error.")

OPENAI_API_KEY loaded (164 characters): sk-p...VmEA


In [3]:
from dotenv import load_dotenv

# Forces python to overwrite any cached key with the one in your .env file
load_dotenv(override=True)

True

In [6]:
"""
Phase 2 triple extraction: OpenAI models only.

Runs the Phase 2 extraction prompt against a fixed sample of chunks for:
    - gpt-5.6-terra
    - gpt-5.6-luna
    - gpt-5.5-pro

Produces two outputs:
    - phase2_openai_raw.xlsx       one row per (model, chunk) with raw JSON
    - phase2_openai_triples.xlsx   one row per extracted triple, flattened

Requires a .env file (not committed, not shared) with:
    OPENAI_API_KEY=...

Usage:
    python run_phase2_openai.py

If you get a 401 "Incorrect API key" error, the key in .env is wrong or
stale. Generate a fresh one at https://platform.openai.com/account/api-keys
and confirm it starts with the correct project prefix (sk-proj-...) for
the project you intend to bill.

Model-specific quirks handled here:
    - gpt-5.6-terra / gpt-5.6-luna: reject temperature=0 (only the default,
      1, is accepted), so temperature is omitted for these two.
    - gpt-5.5-pro: not available via /v1/chat/completions at all, routed
      through /v1/responses (client.responses.create) instead.
"""

import os
import re
import json
import time
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise RuntimeError("OPENAI_API_KEY not found. Add it to your .env file.")

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------

CHUNKS_PATH = r"C:\Users\olagunju\OneDrive\KSU PROJECT\SHOLA_KSU_PUBLISHED_PAPERS\Data-Minning\NER-PROJECT\PDF_PREPROCESSING_INTO_CHUNKS\chunks_all.parquet"
PROMPT_PATH = "phase2_extraction_prompt.md"
SAMPLE_PATH = "sampled_30_chunks.csv"  # reuse the same 30 chunks as the Gemini run
N_CHUNKS = 30
RANDOM_SEED = 42
OUTPUT_DIR = Path("model_comparison_output")
OUTPUT_DIR.mkdir(exist_ok=True)

# Pricing per 1M tokens (input, output), USD, as of Aug 2026.
MODEL_PRICING = {
    "gpt-5.6-terra": (2.50, 15.00),
    "gpt-5.6-luna": (1.00, 6.00),
    "gpt-5.5-pro": (30.00, 180.00),
}

OPENAI_MODELS = ["gpt-5.6-terra", "gpt-5.6-luna", "gpt-5.5-pro"]

# gpt-5.6-terra / gpt-5.6-luna: reasoning-tier models that reject an explicit
# temperature other than the default (1), so temperature is simply omitted
# for them rather than set to 0.
#
# gpt-5.5-pro: not served via /v1/chat/completions at all. it only supports
# the Responses API (client.responses.create), confirmed by the 404 telling
# us to use v1/completions/Responses instead. Routed separately below.
RESPONSES_API_MODELS = {"gpt-5.5-pro"}
NO_TEMPERATURE_MODELS = {"gpt-5.6-terra", "gpt-5.6-luna"}


# ---------------------------------------------------------------------------
# Fail fast on a bad key, before spending 90 calls finding out
# ---------------------------------------------------------------------------

def verify_openai_key():
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)
    try:
        client.models.list()
    except Exception as e:
        raise RuntimeError(
            f"OpenAI API key check failed before running any chunks: {e}\n"
            "Fix OPENAI_API_KEY in .env and re-run."
        )
    print("OpenAI API key verified.")


# ---------------------------------------------------------------------------
# Load prompt + chunk sample
# ---------------------------------------------------------------------------

def load_system_prompt(path: str) -> str:
    text = Path(path).read_text()
    text = re.sub(r"^#\s+Phase 2.*\n", "", text, count=1)
    return text.strip()


def get_chunk_sample() -> pd.DataFrame:
    """Reuse the exact same 30 chunks from the Gemini run if the CSV is
    available, so both provider runs stay directly comparable.
    """
    if Path(SAMPLE_PATH).exists():
        return pd.read_csv(SAMPLE_PATH)
    df = pd.read_parquet(CHUNKS_PATH)
    df = df[df["chunk_text"].str.strip().str.len() > 200].reset_index(drop=True)
    return df.sample(n=N_CHUNKS, random_state=RANDOM_SEED).reset_index(drop=True)


SYSTEM_PROMPT = load_system_prompt(PROMPT_PATH)
sample_df = get_chunk_sample()
print(f"Using {len(sample_df)} chunks from {SAMPLE_PATH if Path(SAMPLE_PATH).exists() else 'fresh sample'}")


def build_user_message(row: pd.Series) -> str:
    return (
        f"pmid: {row['doi']}\n"
        f"section: {row['section']}\n\n"
        f"chunk_text:\n{row['chunk_text']}"
    )


# ---------------------------------------------------------------------------
# JSON parsing helper
# ---------------------------------------------------------------------------

def parse_triples(raw_text: str):
    cleaned = re.sub(r"^```(?:json)?\s*|\s*```$", "", raw_text.strip(), flags=re.MULTILINE)
    try:
        parsed = json.loads(cleaned)
        if isinstance(parsed, dict):
            parsed = [parsed]
        return parsed, None
    except json.JSONDecodeError as e:
        return None, f"JSON parse error: {e}"


# ---------------------------------------------------------------------------
# OpenAI runner
# ---------------------------------------------------------------------------

def call_chat_completions(client, model_name: str, user_msg: str):
    """Standard path for gpt-5.6-terra and gpt-5.6-luna."""
    kwargs = dict(
        model=model_name,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_msg},
        ],
    )
    if model_name not in NO_TEMPERATURE_MODELS:
        kwargs["temperature"] = 0
    response = client.chat.completions.create(**kwargs)
    raw_output = response.choices[0].message.content
    usage = response.usage
    input_tokens = usage.prompt_tokens if usage else None
    output_tokens = usage.completion_tokens if usage else None
    return raw_output, input_tokens, output_tokens


def call_responses_api(client, model_name: str, user_msg: str):
    """gpt-5.5-pro only supports the Responses API, not chat.completions."""
    response = client.responses.create(
        model=model_name,
        instructions=SYSTEM_PROMPT,
        input=user_msg,
    )
    raw_output = response.output_text
    usage = getattr(response, "usage", None)
    input_tokens = getattr(usage, "input_tokens", None) if usage else None
    output_tokens = getattr(usage, "output_tokens", None) if usage else None
    return raw_output, input_tokens, output_tokens


def run_openai_model(model_name: str, df: pd.DataFrame) -> pd.DataFrame:
    from openai import OpenAI

    client = OpenAI(api_key=OPENAI_API_KEY)
    rows = []

    for i, row in df.iterrows():
        user_msg = build_user_message(row)
        start = time.time()
        try:
            if model_name in RESPONSES_API_MODELS:
                raw_output, input_tokens, output_tokens = call_responses_api(
                    client, model_name, user_msg
                )
            else:
                raw_output, input_tokens, output_tokens = call_chat_completions(
                    client, model_name, user_msg
                )
            elapsed = time.time() - start
            triples, err = parse_triples(raw_output)

            rows.append({
                "model": model_name,
                "chunk_id": row["id"],
                "doi": row["doi"],
                "section": row["section"],
                "raw_output": raw_output,
                "n_triples": len(triples) if triples is not None else None,
                "parse_error": err,
                "input_tokens": input_tokens,
                "output_tokens": output_tokens,
                "latency_sec": round(elapsed, 2),
            })
            print(f"  [{model_name}] {i+1}/{len(df)} ok "
                  f"({len(triples) if triples else 0} triples, {elapsed:.1f}s)")
        except Exception as e:
            rows.append({
                "model": model_name,
                "chunk_id": row["id"],
                "doi": row["doi"],
                "section": row["section"],
                "raw_output": None,
                "n_triples": None,
                "parse_error": f"API error: {e}",
                "input_tokens": None,
                "output_tokens": None,
                "latency_sec": None,
            })
            print(f"  [{model_name}] {i+1}/{len(df)} FAILED: {e}")

    return pd.DataFrame(rows)


# ---------------------------------------------------------------------------
# Cost estimate
# ---------------------------------------------------------------------------

def add_cost_column(df: pd.DataFrame) -> pd.DataFrame:
    def _cost(r):
        if pd.isna(r["input_tokens"]) or pd.isna(r["output_tokens"]):
            return None
        in_price, out_price = MODEL_PRICING[r["model"]]
        return (r["input_tokens"] / 1_000_000 * in_price) + (r["output_tokens"] / 1_000_000 * out_price)
    df["est_cost_usd"] = df.apply(_cost, axis=1)
    return df


# ---------------------------------------------------------------------------
# Flatten raw_output JSON into one row per triple
# ---------------------------------------------------------------------------

TRIPLE_FIELDS = [
    "pmid", "source", "source_type", "material", "interaction", "target",
    "target_type", "compared_property", "reported_value", "claim_status",
    "flagged_phrase", "corresponding_sentence",
]


def flatten_triples(raw_df: pd.DataFrame) -> pd.DataFrame:
    flat_rows = []
    for _, row in raw_df.iterrows():
        if pd.isna(row["raw_output"]):
            continue
        triples, err = parse_triples(row["raw_output"])
        if triples is None:
            continue
        for t in triples:
            flat_row = {
                "model": row["model"],
                "chunk_id": row["chunk_id"],
                "doi": row["doi"],
                "section": row["section"],
            }
            for field in TRIPLE_FIELDS:
                flat_row[field] = t.get(field, "")
            flat_rows.append(flat_row)
    return pd.DataFrame(flat_rows)


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

def main():
    verify_openai_key()

    all_results = []
    for model_name in OPENAI_MODELS:
        print(f"\nRunning {model_name} on {len(sample_df)} chunks...")
        result_df = run_openai_model(model_name, sample_df)
        all_results.append(result_df)

    combined = pd.concat(all_results, ignore_index=True)
    combined = add_cost_column(combined)

    raw_path = OUTPUT_DIR / "phase2_openai_raw.xlsx"
    combined.to_excel(raw_path, index=False)
    print(f"\nRaw comparison results -> {raw_path}")

    triples_df = flatten_triples(combined)
    triples_path = OUTPUT_DIR / "phase2_openai_triples.xlsx"
    triples_df.to_excel(triples_path, index=False)
    print(f"Flattened triples ({len(triples_df)} rows) -> {triples_path}")

    summary = (
        combined.groupby("model")
        .agg(
            chunks_run=("chunk_id", "count"),
            chunks_failed=("parse_error", lambda x: x.notna().sum()),
            avg_triples_per_chunk=("n_triples", "mean"),
            total_input_tokens=("input_tokens", "sum"),
            total_output_tokens=("output_tokens", "sum"),
            total_est_cost_usd=("est_cost_usd", "sum"),
            avg_latency_sec=("latency_sec", "mean"),
        )
        .reset_index()
    )
    summary["est_cost_per_1000_chunks_usd"] = (
        summary["total_est_cost_usd"] / len(sample_df) * 1000
    )
    summary_path = OUTPUT_DIR / "phase2_openai_summary.xlsx"
    summary.to_excel(summary_path, index=False)
    print(f"Summary -> {summary_path}")
    print("\n" + summary.to_string(index=False))
    print(
        "\nNote: avg_triples_per_chunk and chunks_failed are volume/parse-success "
        "signals only, not accuracy. Open phase2_openai_triples.xlsx and read "
        "against the source chunks to judge actual quality."
    )


if __name__ == "__main__":
    main()

Using 30 chunks from fresh sample
OpenAI API key verified.

Running gpt-5.6-terra on 30 chunks...
  [gpt-5.6-terra] 1/30 ok (15 triples, 23.5s)
  [gpt-5.6-terra] 2/30 ok (17 triples, 28.1s)
  [gpt-5.6-terra] 3/30 ok (24 triples, 40.3s)
  [gpt-5.6-terra] 4/30 ok (8 triples, 10.9s)
  [gpt-5.6-terra] 5/30 ok (0 triples, 0.5s)
  [gpt-5.6-terra] 6/30 ok (8 triples, 18.8s)
  [gpt-5.6-terra] 7/30 ok (18 triples, 39.9s)
  [gpt-5.6-terra] 8/30 ok (11 triples, 16.6s)
  [gpt-5.6-terra] 9/30 ok (9 triples, 14.0s)
  [gpt-5.6-terra] 10/30 ok (4 triples, 6.9s)
  [gpt-5.6-terra] 11/30 ok (11 triples, 20.2s)
  [gpt-5.6-terra] 12/30 ok (19 triples, 24.7s)
  [gpt-5.6-terra] 13/30 ok (7 triples, 14.5s)
  [gpt-5.6-terra] 14/30 ok (0 triples, 2.8s)
  [gpt-5.6-terra] 15/30 ok (8 triples, 13.4s)
  [gpt-5.6-terra] 16/30 ok (20 triples, 28.2s)
  [gpt-5.6-terra] 17/30 ok (13 triples, 24.2s)
  [gpt-5.6-terra] 18/30 ok (21 triples, 22.0s)
  [gpt-5.6-terra] 19/30 ok (0 triples, 4.3s)
  [gpt-5.6-terra] 20/30 ok (2 t